In [0]:
# Azure ADLS Gen2 Configuration
storage_account_name = "<STORAGE_ACCOUNT_NAME>"
container_name = "bronze"
client_id = "<CLIENT_ID>"
tenant_id = "<TENANT_ID>"
client_secret = "<CLIENT_SECRET>"

# Configure OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder.appName("ECommerceDataPipeline").getOrCreate()

In [0]:
bronze_base_path = "abfss://bronze@ecomadlskyle.dfs.core.windows.net/"
usersDF = spark.read.format("delta").load(f"{bronze_base_path}users")

In [0]:
# Normalize country codes to uppercase
usersDF = usersDF.withColumn("countrycode", upper(col("countryCode")))

# Map language codes to full names (case-insensitive)
usersDF = usersDF.withColumn("language_full", 
    expr("CASE WHEN upper(language) = 'EN' THEN 'English' " +
         "WHEN upper(language) = 'FR' THEN 'French' " +
         "WHEN upper(language) = 'DE' THEN 'German' " +
         "ELSE 'Other' END"))

# Standardize gender values
usersDF = usersDF.withColumn("gender",
    when(col("gender").isin("M", "m", "Male", "male"), "M")
    .when(col("gender").isin("F", "f", "Female", "female"), "F")
    .otherwise("Unknown"))

In [0]:
# Standardize civility titles (Mme, Ms, Mrs -> Ms)
usersDF = usersDF.withColumn("civilitytitle_clean",
    regexp_replace("civilitytitle", "(Mme|Ms|Mrs)", "Ms"))

In [0]:
# Calculate years since last login
usersDF = usersDF.withColumn("years_since_last_login", col("dayssincelastlogin") / 365)

In [0]:
# Calculate account age and categorize
usersDF = usersDF.withColumn("account_age_years", round(col("seniority") / 365, 2))
usersDF = usersDF.withColumn("account_age_group",
    when(col("account_age_years") < 1, "New")
    .when((col("account_age_years") >= 1) & (col("account_age_years") < 3), "Intermediate")
    .otherwise("Experienced"))

In [0]:
# Add current year column for time-based analysis
usersDF = usersDF.withColumn("current_year", year(current_date()))

In [0]:
# Create composite user descriptor
usersDF = usersDF.withColumn("user_descriptor",
    concat(col("gender"), lit("_"),
           col("countrycode"), lit("_"),
           expr("substring(civilitytitle_clean, 1, 3)"), lit("_"),
           col("language_full")))

In [0]:
# Flag users with long civility titles
usersDF = usersDF.withColumn("flag_long_title", length(col("civilitytitle")) > 10)

In [0]:
# Cast status columns to Boolean
usersDF = usersDF.withColumn("hasanyapp", col("hasanyapp").cast("boolean"))
usersDF = usersDF.withColumn("hasandroidapp", col("hasandroidapp").cast("boolean"))
usersDF = usersDF.withColumn("hasiosapp", col("hasiosapp").cast("boolean"))
usersDF = usersDF.withColumn("hasprofilepicture", col("hasprofilepicture").cast("boolean"))

In [0]:
# Cast days since last login to Integer (default to 0 if null)
usersDF = usersDF.withColumn("dayssincelastlogin",
    when(col("dayssincelastlogin").isNotNull(),
         col("dayssincelastlogin").cast(IntegerType()))
    .otherwise(0))

In [0]:
usersDF.show(5)

+--------------------+----+----------+--------+-----------------+---------------+-------------------+--------------+------------+----------------+--------------+--------------+------+----------------+-------------+---------+-------------+---------+-----------------+------------------+---------+-----------------+----------------+-----------+-------------+-------------------+----------------------+-----------------+-----------------+------------+-------------------+---------------+
|      identifierHash|type|   country|language|socialnbfollowers|socialnbfollows|socialProductsLiked|productsListed|productsSold|productspassrate|productsWished|productsBought|gender|civilityGenderId|civilityTitle|hasanyapp|hasandroidapp|hasiosapp|hasprofilepicture|dayssincelastlogin|seniority|seniorityasmonths|seniorityasyears|countryCode|language_full|civilitytitle_clean|years_since_last_login|account_age_years|account_age_group|current_year|    user_descriptor|flag_long_title|
+--------------------+----+---

In [0]:
usersDF.select("account_age_group").show(5)

+-----------------+
|account_age_group|
+-----------------+
|      Experienced|
|      Experienced|
|      Experienced|
|      Experienced|
|      Experienced|
+-----------------+
only showing top 5 rows


In [0]:
silver_base_path = 'abfss://silver@ecomadlskyle.dfs.core.windows.net/'

In [0]:
# Deduplicate by identifierHash and write to Silver
usersDF = usersDF.dropDuplicates(["identifierHash"])

usersDF.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{silver_base_path}/users")

In [0]:
buyersDF = spark.read.format("delta").load(f"{bronze_base_path}buyers")

In [0]:
# Columns to cast to Integer
integer_columns = [
    'buyers', 'topbuyers', 'femalebuyers', 'malebuyers',
    'topfemalebuyers', 'topmalebuyers', 'totalproductsbought',
    'totalproductswished', 'totalproductsliked'
]

for column_name in integer_columns:
    buyersDF = buyersDF.withColumn(column_name, col(column_name).cast(IntegerType()))

In [0]:
# Columns to cast to Decimal(10,2)
decimal_columns = [
    'topbuyerratio', 'femalebuyersratio', 'topfemalebuyersratio',
    'boughtperwishlistratio', 'boughtperlikeratio', 'topboughtperwishlistratio',
    'topboughtperlikeratio', 'meanproductsbought', 'meanproductswished',
    'meanproductsliked', 'topmeanproductsbought', 'topmeanproductswished',
    'topmeanproductsliked', 'meanofflinedays', 'topmeanofflinedays',
    'meanfollowers', 'meanfollowing', 'topmeanfollowers', 'topmeanfollowing'
]

for column_name in decimal_columns:
    buyersDF = buyersDF.withColumn(column_name, col(column_name).cast(DecimalType(10, 2)))

In [0]:
# Normalize country names (capitalize first letter)
buyersDF = buyersDF.withColumn("country", initcap(col("country")))

# Fill nulls in integer columns with 0
for col_name in integer_columns:
    buyersDF = buyersDF.fillna({col_name: 0})

# Calculate female-to-male buyer ratio
buyersDF = buyersDF.withColumn("female_to_male_ratio",
    round(col("femalebuyers") / (col("malebuyers") + 1), 2))

# Calculate wishlist-to-purchase ratio
buyersDF = buyersDF.withColumn("wishlist_to_purchase_ratio", 
    round(col("totalproductswished") / (col("totalproductsbought") + 1), 2))

# Flag high-engagement countries
high_engagement_threshold = 0.5
buyersDF = buyersDF.withColumn("high_engagement",
    when(col("boughtperwishlistratio") > high_engagement_threshold, "True")
    .otherwise("False"))

# Flag growing female markets
buyersDF = buyersDF.withColumn("growing_female_market", 
    when(col("femalebuyersratio") > col("topfemalebuyersratio"), True)
    .otherwise(False))

In [0]:
buyersDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base_path}/buyers")

In [0]:
sellersDF = spark.read.format("delta").load(f"{bronze_base_path}sellers")

In [0]:
# Cast all seller columns to appropriate types
sellersDF = sellersDF \
    .withColumn("nbsellers", col("nbsellers").cast(IntegerType())) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meansellerpassrate", col("meansellerpassrate").cast(DecimalType(10, 2))) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("meanproductsbought", col("meanproductsbought").cast(DecimalType(10, 2))) \
    .withColumn("meanproductswished", col("meanproductswished").cast(DecimalType(10, 2))) \
    .withColumn("meanproductsliked", col("meanproductsliked").cast(DecimalType(10, 2))) \
    .withColumn("totalbought", col("totalbought").cast(IntegerType())) \
    .withColumn("totalwished", col("totalwished").cast(IntegerType())) \
    .withColumn("totalproductsliked", col("totalproductsliked").cast(IntegerType())) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollows", col("meanfollows").cast(DecimalType(10, 2))) \
    .withColumn("percentofappusers", col("percentofappusers").cast(DecimalType(10, 2))) \
    .withColumn("percentofiosusers", col("percentofiosusers").cast(DecimalType(10, 2))) \
    .withColumn("meanseniority", col("meanseniority").cast(DecimalType(10, 2)))

In [0]:
# Normalize country (capitalize) and sex (uppercase)
sellersDF = sellersDF.withColumn("country", initcap(col("country"))) \
                     .withColumn("sex", upper(col("sex")))

# Filter out rows with literal 'None' as country value
sellersDF = sellersDF.filter(~(col("country").isin("None", "none", "NONE")))

# Categorize seller market size
sellersDF = sellersDF.withColumn("seller_size_category", 
    when(col("nbsellers") < 500, "Small")
    .when((col("nbsellers") >= 500) & (col("nbsellers") < 2000), "Medium")
    .otherwise("Large"))

# Calculate mean products listed per seller
sellersDF = sellersDF.withColumn("mean_products_listed_per_seller", 
    round(col("totalproductslisted") / col("nbsellers"), 2))

# Flag markets with high seller pass rate
sellersDF = sellersDF.withColumn("high_seller_pass_rate", 
    when(col("meansellerpassrate") > 0.75, "High")
    .otherwise("Normal"))

# Fill null pass rates with column average
mean_pass_rate_row = sellersDF.select(round(avg("meansellerpassrate"), 2).alias("avg_pass_rate")).collect()
mean_pass_rate = mean_pass_rate_row[0]["avg_pass_rate"] if mean_pass_rate_row else None

if mean_pass_rate is not None:
    sellersDF = sellersDF.withColumn("meansellerpassrate", 
        when(col("meansellerpassrate").isNull(), mean_pass_rate)
        .otherwise(col("meansellerpassrate")))

In [0]:
sellersDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base_path}/sellers")

In [0]:
countriesDF = spark.read.format("delta").load(f"{bronze_base_path}countries")

In [0]:
# Cast all country columns to appropriate types
countriesDF = countriesDF \
    .withColumn("sellers", col("sellers").cast(IntegerType())) \
    .withColumn("topsellers", col("topsellers").cast(IntegerType())) \
    .withColumn("topsellerratio", col("topsellerratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellersratio", col("femalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("topfemalesellersratio", col("topfemalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellers", col("femalesellers").cast(IntegerType())) \
    .withColumn("malesellers", col("malesellers").cast(IntegerType())) \
    .withColumn("topfemalesellers", col("topfemalesellers").cast(IntegerType())) \
    .withColumn("topmalesellers", col("topmalesellers").cast(IntegerType())) \
    .withColumn("countrysoldratio", col("countrysoldratio").cast(DecimalType(10, 2))) \
    .withColumn("bestsoldratio", col("bestsoldratio").cast(DecimalType(10, 2))) \
    .withColumn("toptotalproductssold", col("toptotalproductssold").cast(IntegerType())) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("toptotalproductslisted", col("toptotalproductslisted").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("topmeanproductssold", col("topmeanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("topmeanproductslisted", col("topmeanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanofflinedays", col("meanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("topmeanofflinedays", col("topmeanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowing", col("meanfollowing").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowers", col("topmeanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowing", col("topmeanfollowing").cast(DecimalType(10, 2)))

In [0]:
# Normalize country name
countriesDF = countriesDF.withColumn("country", initcap(col("country")))

# Calculate top seller ratio (with division-by-zero guard)
countriesDF = countriesDF.withColumn("top_seller_ratio", 
    when(col("sellers") > 0, round(col("topsellers") / col("sellers"), 2))
    .otherwise(lit(None)))

# Flag countries with high female seller ratio (>50%)
countriesDF = countriesDF.withColumn("high_female_seller_ratio", 
    when(col("femalesellersratio") > 0.5, True)
    .otherwise(False))

# Calculate performance indicator (sold/listed ratio)
countriesDF = countriesDF.withColumn("performance_indicator", 
    round(col("toptotalproductssold") / (col("toptotalproductslisted") + 1), 2))

# Flag high-performance countries
performance_threshold = 0.8
countriesDF = countriesDF.withColumn("high_performance", 
    when(col("performance_indicator") > performance_threshold, True)
    .otherwise(False))

# Categorize activity level by offline days
countriesDF = countriesDF.withColumn("activity_level", 
    when(col("meanofflinedays") < 30, "Highly Active")
    .when((col("meanofflinedays") >= 30) & (col("meanofflinedays") < 60), "Moderately Active")
    .otherwise("Low Activity"))

In [0]:
countriesDF.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base_path}/countries")

In [0]:
# ============================================================
# SILVER LAYER DATA QUALITY VERIFICATION
# ============================================================
from pyspark.sql.functions import col, count, when, lit, isnan
from pyspark.sql.types import DoubleType, FloatType
import builtins

silver_base_path_v = 'abfss://silver@ecomadlskyle.dfs.core.windows.net/'
su = spark.read.format("delta").load(f"{silver_base_path_v}users")
sb = spark.read.format("delta").load(f"{silver_base_path_v}buyers")
ss = spark.read.format("delta").load(f"{silver_base_path_v}sellers")
sc = spark.read.format("delta").load(f"{silver_base_path_v}countries")

print("="*60)
print("  SILVER LAYER DATA QUALITY REPORT")
print("="*60)

# --- 1. ROW COUNTS ---
users_count = su.count()
buyers_count = sb.count()
sellers_count = ss.count()
countries_count = sc.count()
print(f"\n1. ROW COUNTS")
print(f"   Users:     {users_count:,}")
print(f"   Buyers:    {buyers_count:,}")
print(f"   Sellers:   {sellers_count:,}")
print(f"   Countries: {countries_count:,}")

# --- 2. USERS DEDUP CHECK ---
distinct_users = su.select("identifierHash").distinct().count()
print(f"\n2. USERS DEDUP CHECK")
print(f"   Total rows:            {users_count:,}")
print(f"   Distinct identifiers:  {distinct_users:,}")
print(f"   Duplicates:            {users_count - distinct_users:,}")
print(f"   Status: {'PASS - No duplicates' if users_count == distinct_users else 'FAIL - Duplicates found'}")

# --- 3. COUNTRY CONSISTENCY ---
users_countries = set(r['country'] for r in su.select("country").distinct().collect() if r['country'])
buyers_countries = set(r['country'] for r in sb.select("country").distinct().collect() if r['country'])
sellers_countries = set(r['country'] for r in ss.select("country").distinct().collect() if r['country'])
countries_countries = set(r['country'] for r in sc.select("country").distinct().collect() if r['country'])

print(f"\n3. COUNTRY VALUE CONSISTENCY")
print(f"   Distinct countries - Users: {len(users_countries)}, Buyers: {len(buyers_countries)}, "
      f"Sellers: {len(sellers_countries)}, Countries: {len(countries_countries)}")

# Check for overlap
overlap_b = users_countries & buyers_countries
overlap_s = users_countries & sellers_countries
overlap_c = users_countries & countries_countries
print(f"   Users ∩ Buyers:    {len(overlap_b)} matches")
print(f"   Users ∩ Sellers:   {len(overlap_s)} matches")
print(f"   Users ∩ Countries: {len(overlap_c)} matches")

# Countries only in users (won't join)
only_users_vs_buyers = users_countries - buyers_countries
only_users_vs_countries = users_countries - countries_countries
if only_users_vs_countries:
    print(f"   Countries in Users but NOT in Countries table ({len(only_users_vs_countries)}): "
          f"{sorted(list(only_users_vs_countries))[:5]}...")

# --- 4. SELLERS 'None' FILTER CHECK ---
none_count = ss.filter(col("country").isin("None", "none", "NONE")).count()
print(f"\n4. SELLERS 'None' COUNTRY FILTER")
print(f"   Rows with 'None' country: {none_count}")
print(f"   Status: {'PASS - No None values' if none_count == 0 else 'FAIL - None values remain'}")

# --- 5. SELLERS GRANULARITY (join explosion risk) ---
sellers_per_country = ss.groupBy("country").count().orderBy(col("count").desc())
max_rows = sellers_per_country.first()
print(f"\n5. SELLERS GRANULARITY (country+sex)")
print(f"   Max rows per country: {max_rows['count']} ('{max_rows['country']}')")
print(f"   Note: If >1 row per country, Gold layer join will multiply user rows")

# --- 6. COUNTRIES DIVISION-BY-ZERO CHECK ---
zero_sellers = sc.filter(col("sellers") == 0).count()
null_ratio = sc.filter(col("top_seller_ratio").isNull()).count()
print(f"\n6. COUNTRIES DIVISION-BY-ZERO GUARD")
print(f"   Countries with sellers=0: {zero_sellers}")
print(f"   top_seller_ratio is null:  {null_ratio}")
print(f"   Status: {'PASS - Zero-guard working' if zero_sellers == null_ratio else 'CHECK - Mismatch'}")

# --- 7. NULL SUMMARY PER TABLE ---
print(f"\n7. NULL SUMMARY (columns with >10% nulls)")
for name, df in [("Users", su), ("Buyers", sb), ("Sellers", ss), ("Countries", sc)]:
    total = df.count()
    high_null_cols = []
    for c in df.columns:
        null_ct = df.filter(col(c).isNull()).count()
        pct = (null_ct / total) * 100 if total > 0 else 0
        if pct > 10:
            high_null_cols.append(f"{c}({pct:.0f}%)")
    if high_null_cols:
        print(f"   {name}: {', '.join(high_null_cols)}")
    else:
        print(f"   {name}: All columns < 10% nulls")

# --- 8. USERS LANGUAGE FIX CHECK ---
lang_counts = su.groupBy("language_full").count().orderBy(col("count").desc()).collect()
print(f"\n8. USERS LANGUAGE MAPPING")
for r in lang_counts:
    print(f"   {r['language_full']}: {r['count']:,}")

print(f"\n{'='*60}")
print("  VERIFICATION COMPLETE")
print(f"{'='*60}")

  SILVER LAYER DATA QUALITY REPORT

1. ROW COUNTS
   Users:     98,913
   Buyers:    62
   Sellers:   70
   Countries: 19

2. USERS DEDUP CHECK
   Total rows:            98,913
   Distinct identifiers:  98,913
   Duplicates:            0
   Status: PASS - No duplicates

3. COUNTRY VALUE CONSISTENCY
   Distinct countries - Users: 200, Buyers: 62, Sellers: 47, Countries: 19
   Users ∩ Buyers:    56 matches
   Users ∩ Sellers:   43 matches
   Users ∩ Countries: 16 matches
   Countries in Users but NOT in Countries table (184): ['Afghanistan', 'Afrique du Sud', 'Albanie', 'Algérie', 'Andorre']...

4. SELLERS 'None' COUNTRY FILTER
   Rows with 'None' country: 0
   Status: PASS - No None values

5. SELLERS GRANULARITY (country+sex)
   Max rows per country: 2 ('Autriche')
   Note: If >1 row per country, Gold layer join will multiply user rows

6. COUNTRIES DIVISION-BY-ZERO GUARD
   Countries with sellers=0: 0
   top_seller_ratio is null:  0
   Status: PASS - Zero-guard working

7. NULL SUMMAR